# 01 — Data Cleaning

Load raw parquet files, apply per-dataset cleaning functions, and save cleaned parquets.
All cleaning logic lives in `scripts/cleaning_functions.py`.

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'src').exists():
        project_root = candidate
        break
sys.path.insert(0, str(project_root))

import pandas as pd
from src.scripts.data_utils import load_raw_parquets, PARQUET_CLEAN
from src.scripts.cleaning_functions import clean_all
from src.scripts.export_data import save_cleaned_parquets

ModuleNotFoundError: No module named 'src'

## Load Raw Parquet Files

In [ ]:
tables = load_raw_parquets()

Loaded hpa_rna: (24315372, 6)
Loaded depmap_expr: (1495, 53961)
Loaded geo_expr: (19914, 3268)
Loaded proteomics: (375, 12559)
Loaded fusions: (184237, 30)
Loaded mutations: (1066869, 70)
Loaded cellosaurus: (152231, 17)
Loaded depmap_profiles: (3830, 5)
Loaded sample_info: (1840, 29)
Loaded geo_info: (3267, 23)
Loaded hpa_desc: (1206, 7)
Loaded metabolomics: (928, 227)
Loaded mirna: (734, 956)
Loaded signatures: (3021, 12)


## Apply Cleaning Functions

Each function handles:
- Lowercase column names and string values
- Strip / collapse whitespace
- Strip Ensembl version suffixes (e.g. `ENSG00000000003.15` → `ENSG00000000003`)
- Dataset-specific transformations (column splits, renames, header parsing)

In [ ]:
cleaned = clean_all(tables)
print(f"Cleaned {len(cleaned)} datasets.")

Cleaned 15 datasets.


## Inspect Individual Cleaned Tables

In [ ]:
# Quick shape check on all cleaned tables
for name, df in cleaned.items():
    print(f"{name:25s}: {df.shape[0]:>10,} rows x {df.shape[1]:>6,} cols")

hpa_rna                  : 24,315,372 rows x      6 cols
depmap_expr              :      1,495 rows x 53,961 cols
geo_expr                 :     19,914 rows x  3,268 cols
proteomics               :        375 rows x 12,559 cols
protein_map              :     12,558 rows x      3 cols
fusions                  :    184,237 rows x     32 cols
mutations                :  1,066,869 rows x     70 cols
cellosaurus              :    152,231 rows x     17 cols
depmap_profiles          :      3,830 rows x      5 cols
sample_info              :      1,840 rows x     29 cols
geo_info                 :      3,267 rows x     23 cols
hpa_desc                 :      1,206 rows x      7 cols
metabolomics             :        928 rows x    227 cols
mirna                    :        734 rows x    956 cols
signatures               :      3,021 rows x     12 cols


In [ ]:
# Spot-check: HPA RNA — verify gene column has no version suffixes
cleaned["hpa_rna"].head()

,gene,gene name,cell line,tpm,ptpm,ntpm
0,ensg00000000003,tspan6,143b,22.0,27.6,25.9
1,ensg00000000003,tspan6,22rv1,2.8,3.6,2.7
2,ensg00000000003,tspan6,23132/87,6.2,7.5,7.5
3,ensg00000000003,tspan6,253j,14.2,18.7,25.4
4,ensg00000000003,tspan6,253j-bv,13.0,17.1,18.5


In [ ]:
cleaned["geo_expr"].head()

,gene,gsm101610,gsm101615,gsm101616,gsm101667,gsm101668,gsm101671,gsm101672,gsm101673,gsm101674,...,gsm960289,gsm960290,gsm960291,gsm960292,gsm960293,gsm960294,gsm960295,gsm960296,gsm960297,gsm960298
0,ensg00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ensg00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ensg00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952
3,ensg00000000457,58.934814,95.068222,94.900459,51.810070,52.327530,50.011375,53.793667,51.172344,52.677490,...,35.778347,57.736485,25.528204,24.593216,23.337006,47.793488,41.268002,64.042831,37.363773,61.027988
4,ensg00000000460,136.418900,257.929169,271.317230,162.090073,160.913849,206.788635,209.966019,134.272583,147.408554,...,14.693585,19.083935,114.056015,137.948593,149.641708,111.793892,98.536613,167.151016,12.268769,24.296593


In [ ]:
# Spot-check: Fusions — verify gene1/gene2 split columns
cleaned["fusions"][["canonicalfusionname", "gene1", "gene1_ens_id", "gene2", "gene2_ens_id"]].head()

,canonicalfusionname,gene1,gene1_ens_id,gene2,gene2_ens_id
0,dlg1--serpini1,dlg1,ensg00000075711,serpini1,.
1,adam17--itgb1bp1,adam17,ensg00000151694,itgb1bp1,ensg00000119185
2,adam17--itgb1bp1,adam17,ensg00000151694,itgb1bp1,ensg00000119185
3,adam17--itgb1bp1,adam17,ensg00000151694,itgb1bp1,ensg00000119185
4,adam17--itgb1bp1,adam17,ensg00000151694,itgb1bp1,ensg00000119185


In [ ]:
# Spot-check: Proteomics protein map
cleaned["protein_map"].head()

,uniprot_id,gene_symbol,original_header
0,a0av96,rbm47,a0av96 (rbm47)
1,a0avf1,ift56,a0avf1 (ift56)
2,a0avg3,tsnare1,a0avg3 (tsnare1)
3,a0avi4,tmem129,a0avi4 (tmem129)
4,a0avk6,e2f8,a0avk6 (e2f8)


## Missing Value Summary After Cleaning

In [ ]:
rows = []
for name, df in cleaned.items():
    missing_pct = df.isnull().sum().sum() / df.size * 100 if df.size else 0
    rows.append({"Dataset": name, "Rows": f"{df.shape[0]:,}", "Cols": df.shape[1], "Missing %": f"{missing_pct:.1f}%"})
pd.DataFrame(rows)

,Dataset,Rows,Cols,Missing %
0,hpa_rna,"24,315,372",6,0.0%
1,depmap_expr,"1,495",53961,0.0%
2,geo_expr,"19,914",3268,0.0%
3,proteomics,375,12559,27.8%
4,protein_map,"12,558",3,0.0%
5,fusions,"184,237",32,0.0%
6,mutations,"1,066,869",70,37.4%
7,cellosaurus,"152,231",17,36.4%
8,depmap_profiles,"3,830",5,10.3%
9,sample_info,"1,840",29,34.0%


## Save Cleaned DataFrames as Parquet

In [ ]:
save_cleaned_parquets(cleaned, out_dir=PARQUET_CLEAN)

Saved hpa_rna -> hpa_rna_clean.parquet (113.9 MB)
Saved depmap_expr -> depmap_expr_clean.parquet (228.1 MB)
Saved geo_expr -> geo_expr_clean.parquet (518.6 MB)
Saved proteomics -> proteomics_clean.parquet (36.4 MB)
Saved protein_map -> protein_map.parquet (0.3 MB)
Saved fusions -> fusions_clean.parquet (9.3 MB)
Saved mutations -> mutations_clean.parquet (48.0 MB)
Saved cellosaurus -> cellosaurus_clean.parquet (13.4 MB)
Saved depmap_profiles -> depmap_profiles_clean.parquet (0.1 MB)
Saved sample_info -> sample_info_clean.parquet (0.2 MB)
Saved geo_info -> geo_info_clean.parquet (0.1 MB)
Saved hpa_desc -> hpa_desc_clean.parquet (0.0 MB)
Saved metabolomics -> metabolomics_clean.parquet (2.0 MB)
Saved mirna -> mirna_clean.parquet (2.3 MB)
Saved signatures -> signatures_clean.parquet (0.2 MB)
